# 中文分词实践：基于最大匹配的分词方法

最大匹配（Maximum Matching）算法主要包含三类：

- **前向最大匹配（Forward Maximum Matching, FMM）**：从左向右扫描句子，每次选择当前位置起、词典中能匹配上的最长词。
- **后向最大匹配（Backward Maximum Matching, BMM）**：思想相同，区别在于从右向左扫描。
- **双向最大匹配（Bi-directional Maximum Matching, BiMM）**：同时进行前向和后向匹配，当两者结果不同时使用启发式规则（如选择单字词更少、或平均词长更大的一个）决定最终输出。

我们将实现这三种算法，并复现书中表 2.2 给出的例子：

> 句子：**"他是研究生物化学的一位科学家"**
>
> 词典（最大词长假设为 4）：{"他", "是", "研究", "生物化学", "的", "一", "位", "科学家"}

前向最大匹配算法的流程如下：

输入：字符序列 c[1..N]，词典 D，最大词长 L

输出：分词结果
```pseudocode
For i = 1 to N do
    For sub = { c[i : i+L-1], c[i : i+L-2], ..., c[i : i] } do   // 按长度从长到短尝试
        if sub ∈ D then
            print sub
            i = i + len(sub)     // 或 i = j + 1，其中 j 为当前子串的结束位置
            break
        end if
    end for
end for
```

## 1. 定义词典与测试句子

In [1]:
# 书中 2.3.2 节例子所使用的词典
vocab = {"他", "是", "研究", "生物化学", "的", "一", "位", "科学家"}

# 词典中最长词的长度，书中假设为 4
max_len = max(len(w) for w in vocab)
print("词典:", vocab)
print("词典最大词长 L =", max_len)

sentence = "他是研究生物化学的一位科学家"
print("待分词句子:", sentence)


词典: {'科学家', '研究', '一', '他', '的', '生物化学', '位', '是'}
词典最大词长 L = 4
待分词句子: 他是研究生物化学的一位科学家


## 2. 前向最大匹配（Forward Maximum Matching）

算法思路（对应原文描述）：

对句子中的一个位置 `i`，依次考虑子串 `c[i:i+L-1], c[i:i+L-2], ..., c[i:i]`（`L` 为词典最大词长），
一旦某个子串能在词典中匹配上，就输出这个词，并从下一个位置继续扫描；如果所有长度都无法匹配，
则退化为输出单字（保证未登录字符也能被切分出来，避免死循环）。


In [2]:
def forward_maximum_matching(sentence, vocab, max_len, verbose=False):
    """前向最大匹配分词

    参数:
        sentence: 待分词的字符串
        vocab: 词典（集合）
        max_len: 词典中词的最大长度 L
        verbose: 是否打印每一步的扫描过程（用于复现书中表 2.2）
    返回:
        分词结果列表
    """
    result = []
    i = 0
    n = len(sentence)
    step = 0
    while i < n:
        step += 1
        matched_word = None
        # 依次尝试从最长到最短的候选子串
        candidates = []
        for l in range(min(max_len, n - i), 0, -1):
            candidate = sentence[i:i + l]
            candidates.append(candidate)
            if candidate in vocab:
                matched_word = candidate
                break
        if matched_word is None:
            # 词典外的单字，直接作为一个词输出（避免死循环）
            matched_word = sentence[i]

        if verbose:
            print(f"时间步 {step}\t开始位置 {i+1}\t候选匹配: {', '.join(candidates)}\t输出: {matched_word}")

        result.append(matched_word)
        i += len(matched_word)
    return result


print("=== 前向最大匹配过程（复现表 2.2） ===")
fmm_result = forward_maximum_matching(sentence, vocab, max_len, verbose=True)
print()
print("最终分词结果:", " | ".join(fmm_result))


=== 前向最大匹配过程（复现表 2.2） ===
时间步 1	开始位置 1	候选匹配: 他是研究, 他是研, 他是, 他	输出: 他
时间步 2	开始位置 2	候选匹配: 是研究生, 是研究, 是研, 是	输出: 是
时间步 3	开始位置 3	候选匹配: 研究生物, 研究生, 研究	输出: 研究
时间步 4	开始位置 5	候选匹配: 生物化学	输出: 生物化学
时间步 5	开始位置 9	候选匹配: 的一位科, 的一位, 的一, 的	输出: 的
时间步 6	开始位置 10	候选匹配: 一位科学, 一位科, 一位, 一	输出: 一
时间步 7	开始位置 11	候选匹配: 位科学家, 位科学, 位科, 位	输出: 位
时间步 8	开始位置 12	候选匹配: 科学家	输出: 科学家

最终分词结果: 他 | 是 | 研究 | 生物化学 | 的 | 一 | 位 | 科学家


## 3. 后向最大匹配（Backward Maximum Matching）

思路与前向最大匹配相同，只是**从右向左**扫描句子。


In [3]:
def backward_maximum_matching(sentence, vocab, max_len, verbose=False):
    """后向最大匹配分词"""
    result = []
    n = len(sentence)
    j = n  # 当前扫描的结束位置（不含）
    step = 0
    while j > 0:
        step += 1
        matched_word = None
        candidates = []
        for l in range(min(max_len, j), 0, -1):
            candidate = sentence[j - l:j]
            candidates.append(candidate)
            if candidate in vocab:
                matched_word = candidate
                break
        if matched_word is None:
            matched_word = sentence[j - 1]

        if verbose:
            print(f"时间步 {step}\t结束位置 {j}\t候选匹配: {', '.join(candidates)}\t输出: {matched_word}")

        result.append(matched_word)
        j -= len(matched_word)
    result.reverse()
    return result


print("=== 后向最大匹配过程 ===")
bmm_result = backward_maximum_matching(sentence, vocab, max_len, verbose=True)
print()
print("最终分词结果:", " | ".join(bmm_result))


=== 后向最大匹配过程 ===
时间步 1	结束位置 14	候选匹配: 位科学家, 科学家	输出: 科学家
时间步 2	结束位置 11	候选匹配: 学的一位, 的一位, 一位, 位	输出: 位
时间步 3	结束位置 10	候选匹配: 化学的一, 学的一, 的一, 一	输出: 一
时间步 4	结束位置 9	候选匹配: 物化学的, 化学的, 学的, 的	输出: 的
时间步 5	结束位置 8	候选匹配: 生物化学	输出: 生物化学
时间步 6	结束位置 4	候选匹配: 他是研究, 是研究, 研究	输出: 研究
时间步 7	结束位置 2	候选匹配: 他是, 是	输出: 是
时间步 8	结束位置 1	候选匹配: 他	输出: 他

最终分词结果: 他 | 是 | 研究 | 生物化学 | 的 | 一 | 位 | 科学家


## 4. 双向最大匹配（Bi-directional Maximum Matching）

同时执行前向和后向最大匹配。若两者结果相同，直接输出；若不同，采用启发式规则选择更优的结果，
常见规则包括：

1. 优先选择**分词数量更少**的结果（通常意味着切分出的词更长、信息量更大）；
2. 若数量相同，选择**单字词更少**的结果；
3. 若仍然相同，两者可视为等价，任选其一。


In [4]:
def bidirectional_maximum_matching(sentence, vocab, max_len, verbose=False):
    """双向最大匹配分词"""
    fmm = forward_maximum_matching(sentence, vocab, max_len)
    bmm = backward_maximum_matching(sentence, vocab, max_len)

    if verbose:
        print("前向结果:", " | ".join(fmm))
        print("后向结果:", " | ".join(bmm))

    if fmm == bmm:
        if verbose:
            print("前向与后向结果一致，直接采用。")
        return fmm

    # 规则 1：词数更少的优先
    if len(fmm) != len(bmm):
        chosen = fmm if len(fmm) < len(bmm) else bmm
        if verbose:
            print("前向、后向结果词数不同，选择词数较少的一个。")
        return chosen

    # 规则 2：单字词更少的优先
    fmm_single = sum(1 for w in fmm if len(w) == 1)
    bmm_single = sum(1 for w in bmm if len(w) == 1)
    if fmm_single != bmm_single:
        chosen = fmm if fmm_single < bmm_single else bmm
        if verbose:
            print("词数相同，选择单字词更少的一个。")
        return chosen

    # 规则 3：两者视为等价，默认选前向
    if verbose:
        print("两种结果难以区分优劣，默认采用前向匹配结果。")
    return fmm


print("=== 双向最大匹配 ===")
bimm_result = bidirectional_maximum_matching(sentence, vocab, max_len, verbose=True)
print()
print("最终分词结果:", " | ".join(bimm_result))


=== 双向最大匹配 ===
前向结果: 他 | 是 | 研究 | 生物化学 | 的 | 一 | 位 | 科学家
后向结果: 他 | 是 | 研究 | 生物化学 | 的 | 一 | 位 | 科学家
前向与后向结果一致，直接采用。

最终分词结果: 他 | 是 | 研究 | 生物化学 | 的 | 一 | 位 | 科学家


## 6. 算法局限性演示

正如原文所述，基于词典的最大匹配方法"简单、快速、可控、仅依赖词表"，但**对切分歧义和未登录词的处理能力不足**。
下面用 2.3.1 节中提到的交集型切分歧义例子 **"乒乓球拍卖完了"** 进行演示。


In [5]:
# 用于演示歧义问题的词典（不含书中完整表 2.2 词典，另起一组示例词）
ambiguity_vocab = {"乒乓", "球拍", "拍卖", "球", "卖", "完", "了"}
ambiguity_max_len = max(len(w) for w in ambiguity_vocab)

s = "乒乓球拍卖完了"
print("待分词句子:", s)
print()

fmm_amb = forward_maximum_matching(s, ambiguity_vocab, ambiguity_max_len, verbose=True)
print("前向最大匹配结果:", " | ".join(fmm_amb))
print()

bmm_amb = backward_maximum_matching(s, ambiguity_vocab, ambiguity_max_len, verbose=True)
print("后向最大匹配结果:", " | ".join(bmm_amb))

待分词句子: 乒乓球拍卖完了

时间步 1	开始位置 1	候选匹配: 乒乓	输出: 乒乓
时间步 2	开始位置 3	候选匹配: 球拍	输出: 球拍
时间步 3	开始位置 5	候选匹配: 卖完, 卖	输出: 卖
时间步 4	开始位置 6	候选匹配: 完了, 完	输出: 完
时间步 5	开始位置 7	候选匹配: 了	输出: 了
前向最大匹配结果: 乒乓 | 球拍 | 卖 | 完 | 了

时间步 1	结束位置 7	候选匹配: 完了, 了	输出: 了
时间步 2	结束位置 6	候选匹配: 卖完, 完	输出: 完
时间步 3	结束位置 5	候选匹配: 拍卖	输出: 拍卖
时间步 4	结束位置 3	候选匹配: 乓球, 球	输出: 球
时间步 5	结束位置 2	候选匹配: 乒乓	输出: 乒乓
后向最大匹配结果: 乒乓 | 球 | 拍卖 | 完 | 了


可以看到：

- **前向最大匹配**会先贪心匹配到"球拍"，进而将句子切分为：`乒乓 | 球拍 | 卖 | 完 | 了`；
- **后向最大匹配**则会先匹配到"拍卖"，切分为：`乒乓 | 球 | 拍卖 | 完 | 了`。

这正是原文所描述的**交集型切分歧义**（"球拍"和"拍卖"共享交集字"拍"）。由于两种贪心策略的扫描方向不同，
最终得到了不同的、且都"合法"的切分结果，这也说明了仅靠词典和贪心策略无法从根本上解决切分歧义问题，
真正的正确切分往往需要结合上下文语义（如本例中"乒乓球拍卖完了" vs. "乒乓球拍卖完了"实际语境）。


## 7. 未登录词（OOV）演示

若句子中出现词典之外的字符组合（如新词、专有名词等），最大匹配算法只能将其逐字切开，
无法识别为一个整体，这正是原文所述"对于没有在词典中出现的词没有很好的处理方案"的体现。


In [6]:
oov_sentence = "我今天用ChatGPT写代码"
oov_vocab = {"我", "今天", "用", "写", "代码"}
oov_max_len = max(len(w) for w in oov_vocab)

result_oov = forward_maximum_matching(oov_sentence, oov_vocab, oov_max_len, verbose=True)
print()
print("分词结果:", " | ".join(result_oov))
print()
print("可以看到 'ChatGPT' 作为词典之外的未登录词，被逐字符拆开，")
print("无法作为一个完整的词形（Token）被识别出来。")


时间步 1	开始位置 1	候选匹配: 我今, 我	输出: 我
时间步 2	开始位置 2	候选匹配: 今天	输出: 今天
时间步 3	开始位置 4	候选匹配: 用C, 用	输出: 用
时间步 4	开始位置 5	候选匹配: Ch, C	输出: C
时间步 5	开始位置 6	候选匹配: ha, h	输出: h
时间步 6	开始位置 7	候选匹配: at, a	输出: a
时间步 7	开始位置 8	候选匹配: tG, t	输出: t
时间步 8	开始位置 9	候选匹配: GP, G	输出: G
时间步 9	开始位置 10	候选匹配: PT, P	输出: P
时间步 10	开始位置 11	候选匹配: T写, T	输出: T
时间步 11	开始位置 12	候选匹配: 写代, 写	输出: 写
时间步 12	开始位置 13	候选匹配: 代码	输出: 代码

分词结果: 我 | 今天 | 用 | C | h | a | t | G | P | T | 写 | 代码

可以看到 'ChatGPT' 作为词典之外的未登录词，被逐字符拆开，
无法作为一个完整的词形（Token）被识别出来。
